# 03 — Sentence-BERT (M1.5 / M1.6)
`paraphrase-multilingual-MiniLM-L12-v2` (offres en français/anglais/arabe). Comparaison à la baseline sur **les mêmes CV et le même pool**.

In [ ]:
# Se placer à la racine du projet (VS Code, Jupyter ou Colab)
import os
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
print("Racine du projet :", ROOT)

In [ ]:
from cvmatch.config import get_settings
from cvmatch.data import load_jobs, job_texts
from cvmatch.ranking.embeddings import build_embeddings, load_embeddings, embeddings_path
from cvmatch.ranking import SbertRanker, TfidfRanker
from cvmatch.evaluation import load_cvs, load_labels, evaluate_ranker, summarize, delta
from cvmatch.preprocessing.cleaning import clean_text
from cvmatch.preprocessing.pii import mask_pii
s = get_settings()
jobs = load_jobs(s.jobs_path)
if not embeddings_path(s.artifacts_dir, s.sbert_model, True).exists():
    build_embeddings(jobs, s.sbert_model, s.artifacts_dir, cleaning=True)
sbert = SbertRanker.from_model(load_embeddings(jobs, s.sbert_model, s.artifacts_dir, True), s.sbert_model)
tfidf = TfidfRanker(job_texts(jobs, True))

In [ ]:
cvs = load_cvs(s.eval_dir / "cvs")
lab = s.eval_dir / "labels.csv"
if not lab.exists():
    print("⚠ labels PROXY : test technique, pas de résultats à rapporter")
    lab = s.eval_dir / "proxy_labels.csv"
labels = load_labels(lab)
prep = lambda t: clean_text(mask_pii(t))
sbert.rank("warm up", top_k=1)
res = {n: evaluate_ranker(r, jobs, cvs, labels, preprocess=prep) for n, r in [("B0_tfidf", tfidf), ("F1_sbert", sbert)]}
import pandas as pd
summary = pd.DataFrame([summarize(d, n) for n, d in res.items()]); summary

In [ ]:
print("Δ P@3 (SBERT - TF-IDF) =", delta(summary.mean_p_at_k[1], summary.mean_p_at_k[0]))
# Si le delta est nul ou négatif, SBERT ne montre pas de bénéfice sur CE test.